# Ekip YOLO modelleri karşılaştırması · tek model mi, ensemble mı?

**LEVEL-UP AI | ROKETSAN Yapay Zekâ Hackathonu · Aşama 1**

Ekibin `scene_holdout_v2` üzerinde eğitilmiş YOLO modelleri aynı doğrulama kümesinde (1.295 görüntü) ve **aynı yarışma metriğiyle** karşılaştırılır.
Ardından modellerin birleşimleri (ensemble) aynı kümede ölçülür. Test tahmini, doğrulamada kazanan adayla üretilir.

| Model | Eğitim tarifi | Ön bilgi |
|---|---|---|
| **furkan** · `y11m_v2` | YOLO11m, RFS + LR'ye bağlı azalan veri artırımı, MuSGD, 51 epoch (5 saat) | 1280'de 0,7566, TTA@1536 + NMS 0,6 ile 0,7894 (ağırlık kayboldu; bulunursa dahil edilir) |
| **hakan** · `yolo11m_1280_v2` | YOLO11m, Ultralytics varsayılanları, AdamW, 91 epoch (erken durdurma, en iyisi 71. epoch) | 1280'de 0,7739 |
| **ardahan_l** · `yolo11l_sh2_1280_e42` | YOLO11l, cos_lr, `flipud=0.5`, `scale=0.4`, 42 epoch, `last.pt` | – |
| **ardahan_m_rfs** · `yolo11m_sh2_rfs_1280` | YOLO11m, bus ×3 / truck ×2 tekrar, cos_lr, `flipud=0.5`, 30 epoch, `last.pt` | 18 epoch'luk ilk koşu v2 val'de 0,7671 |

Tüm modeller COCO ön-eğitimli, `imgsz=1280` ve aynı `scene_holdout_v2` train kümesiyle (5.176 görüntü) eğitildi. Val görüntüleri hiçbirinin eğitiminde yok.

Değerlendirme ve tahmin kodu `furkan/kaggle/yolo11m_v2/yolo11m.ipynb` ile birebir aynıdır. Böylece buradaki sayılar oradakilerle doğrudan karşılaştırılabilir.

### İçindekiler
1. Ortam ve yapılandırma
2. Veri ve doğrulama ağırlıkları
3. Yarışma metriği ve tahmin fonksiyonları
4. Modellerin kontrolü
5. Doğrulama tahminleri
6. Tek model sonuçları
7. Ensemble
8. Karar
9. Test tahmini ve submission

## Yöntem

| Aday | Açıklama |
|---|---|
| **A · 1280** | Eğitim çözünürlüğünde düz tahmin |
| **C · TTA@1536** | Ultralytics TTA (ölçekler 1,0 / 0,83 / 0,67 + yatay çevirme), 1536'da. Furkan'ın modelinde en iyi yöntem |
| **Ensemble** | Modellerin C yöntemiyle ürettiği kutular birleştirilir, üzerine sınıf bazında NMS uygulanır, görüntü başına en fazla 1.000 kutu tutulur. Mevcut modellerin tüm ikili ve daha büyük kombinasyonları denenir |

- Her aday için NMS IoU eşiği 0,50–0,70 arasında taranır. Model tahminleri 0,70'te üretildiği için yeniden tahmin gerekmez.
- **Karar kuralı:** Ensemble, en iyi tek modeli test ağırlıklı mAP'te en az `MIN_GAIN` (0,005) geçerse seçilir. Geçemezse en iyi tek model seçilir. Bu eşik, ölçüm gürültüsü içinde kalan küçük farkların submission hakkı harcatmasını önler.
- WBF kullanılmıyor, birleştirme yalnızca NMS ile yapılıyor.
- **Önbellek:** Tahminler `preds/` klasörüne yazılır. Notebook yeniden çalıştırıldığında mevcut dosyalar tekrar hesaplanmaz.

<a id="1"></a>
## 1 · Ortam ve yapılandırma

Notebook hem **Kaggle'da** (GPU T4) hem **yerelde** (repo içinden) çalışır.

| | Kaggle | Yerel |
|---|---|---|
| Veri | `roketsan-yolo-v2` dataset'i input olarak eklenir | `furkan/outputs/yolo_v2/` (`furkan/src/prepare_yolo.py` çıktısı) |
| furkan | `yolo11m_v2` notebook'unun çıktısı (`y11m_v2/weights/best.pt`) | `furkan/outputs/weights/furkan_y11m_best.pt` |
| hakan | `hakan_yolo11m.pt` bir dataset olarak eklenir | `~/Downloads/hakan_yolo11m.pt` |
| ardahan_l | `yolo11l_ardahan.pt` bir dataset olarak eklenir | `~/Downloads/yolo11l_ardahan.pt` |
| ardahan_m_rfs | `ardahan_embedding.pt` bir dataset olarak eklenir | `~/Downloads/ardahan_embedding.pt` |

Yollar `<MODEL>_PT` (ör. `ARDAHAN_L_PT`) ve `DATA_DIR` ortam değişkenleriyle değiştirilebilir. Bulunamayan model uyarıyla atlanır.

In [ ]:
import os
import sys
import json
import time
import shutil
import zipfile
import subprocess
from pathlib import Path

T0 = time.time()

# --- Deney ayarları -----------------------------------------------------------------------------
IMGSZ      = 1280                                # A yöntemi (eğitim çözünürlüğü)
HI_IMGSZ   = 1536                                # C yöntemi (TTA)
NMS_SWEEP  = [0.50, 0.55, 0.60, 0.65, 0.70]      # taranan sınıf bazlı NMS IoU eşikleri
MAX_DET    = 1000                                # görüntü başına en fazla kutu (yarışma: en fazla 218 gerçek kutu)
MIN_GAIN   = 0.005                               # ensemble'ın seçilmesi için gereken en az kazanç
DARK_THR   = 55                                  # karanlık görüntü eşiği (64×64 gri ortalama; v2 split ile aynı)
CLASSES    = ["car", "van", "truck", "bus"]
ULTRALYTICS_VERSION = "8.4.163"                  # iki model de bu sürümle eğitildi

# --- Yollar -------------------------------------------------------------------------------------
ON_KAGGLE = Path("/kaggle/input").exists()
INPUT = Path("/kaggle/input")
REPO = (Path(os.environ["REPO_DIR"]) if os.environ.get("REPO_DIR") else
        next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "furkan").is_dir() and (p / "splits").is_dir()), None))
WORK = Path("/kaggle/working") if ON_KAGGLE else REPO / "furkan" / "outputs" / "compare_11m"
PRED_DIR = WORK / "preds"
PRED_DIR.mkdir(parents=True, exist_ok=True)


def log(*args):
    print(f"[{(time.time() - T0) / 60:6.1f} dk]", *args, flush=True)


def walk_files(root: Path):
    """Tüm dosyaları gezer; Kaggle input klasörleri symlink olabildiği için linkleri takip eder (Path.rglob etmez)."""
    for dirpath, _, filenames in os.walk(root, followlinks=True):
        for f in filenames:
            yield Path(dirpath) / f


def find_weight(env, kaggle_suffixes, local_path):
    """Ağırlık dosyasını ortam değişkeni → Kaggle input (yol sonekine göre) → yerel varsayılan sırasıyla arar."""
    if os.environ.get(env):
        return Path(os.environ[env])
    if ON_KAGGLE:
        hits = [p for p in walk_files(INPUT) if any(p.as_posix().endswith(s) for s in kaggle_suffixes)]
        return hits[0] if hits else None
    return local_path


WEIGHTS = {
    "furkan": find_weight("FURKAN_PT", ["y11m_v2/weights/best.pt", "furkan_y11m_best.pt"],
                          REPO / "furkan" / "outputs" / "weights" / "furkan_y11m_best.pt" if REPO else None),
    "hakan":  find_weight("HAKAN_PT", ["hakan_yolo11m.pt", "yolo11m_1280_v2/weights/best.pt", "yolo11m_1280_v2/best.pt"],
                          Path.home() / "Downloads" / "hakan_yolo11m.pt"),
    "ardahan_l": find_weight("ARDAHAN_L_PT", ["yolo11l_ardahan.pt"], Path.home() / "Downloads" / "yolo11l_ardahan.pt"),
    "ardahan_m_rfs": find_weight("ARDAHAN_M_RFS_PT", ["ardahan_embedding.pt"],
                                 Path.home() / "Downloads" / "ardahan_embedding.pt"),
}
for k, p in list(WEIGHTS.items()):
    if p is None or not Path(p).is_file():
        log(f"UYARI: {k} ağırlığı bulunamadı ({p}); bu model atlanıyor. "
            f"Kaggle'da input olarak ekleyin ya da {k.upper()}_PT ile yol verin.")
        del WEIGHTS[k]
    else:
        log(f"{k:7s} ağırlığı: {p}")
assert WEIGHTS, "Hiç ağırlık bulunamadı."
ENSEMBLE = len(WEIGHTS) >= 2
log("mod: " + (f"{len(WEIGHTS)} model + ensemble" if ENSEMBLE else f"tek model ({next(iter(WEIGHTS))}), ensemble atlanıyor"))

In [ ]:
try:
    import ultralytics
    assert ultralytics.__version__ == ULTRALYTICS_VERSION
except (ImportError, AssertionError):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)
    import importlib, ultralytics
    importlib.reload(ultralytics)

import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

pd.set_option("display.float_format", "{:,.4f}".format)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
log(f"ultralytics {ultralytics.__version__} | torch {torch.__version__} | GPU: "
    + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "yok (CPU çok yavaş olur)"))

<a id="2"></a>
## 2 · Veri ve doğrulama ağırlıkları

Kaggle'da `roketsan-yolo-v2` parçaları tek klasörde birleştirilir (11m notebook'uyla aynı yöntem). Yerelde hazır klasör kullanılır.

Test ağırlıklı doğrulama: her val görüntüsüne `w = p_test(katman) / p_val(katman)` ağırlığı verilir (katman = çözünürlük × karanlık/aydınlık).
**Karşılaştırmanın birincil ölçütü test ağırlıklı mAP@0.5'tir.**

In [ ]:
LIST_FILES = {"train.txt", "val.txt", "test.txt", "val_1400x788.txt", "val_dark.txt",
              "val_1400x788_dark.txt", "classes.json"}


def dataset_relpath(p: Path):
    """Bir dosyanın yolo_v2/ içindeki göreli yolu; Kaggle'ın zip açma biçiminden bağımsızdır."""
    parts = p.parts
    if "yolo_v2" in parts:
        return Path(*parts[len(parts) - 1 - parts[::-1].index("yolo_v2") + 1:])
    for k in range(len(parts) - 2, 0, -1):
        if parts[k - 1] in ("images", "labels") and parts[k] in ("train", "test"):
            return Path(*parts[k - 1:])
    if p.name in LIST_FILES:
        return Path(p.name)
    return None


def assemble_dataset(ds: Path):
    """/kaggle/input altındaki veri seti parçalarını ds altında birleştirir (açılmamış zip'ler dahil)."""
    if (ds / "classes.json").exists():
        return 0
    unz, n = WORK / "_unz", 0
    for z in (q for q in walk_files(INPUT) if q.suffix == ".zip"):
        with zipfile.ZipFile(z) as f:
            f.extractall(unz)
    for root in (INPUT, unz):
        if root.exists():
            for p in walk_files(root):
                rel = dataset_relpath(p)
                if rel is not None and not (ds / rel).exists():
                    (ds / rel).parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(p, ds / rel)
                    n += 1
    shutil.rmtree(unz, ignore_errors=True)
    assert (ds / "classes.json").exists(), "Veri seti bulunamadı: 'Add Input' → roketsan-yolo-v2 ekli mi?"
    return n


if os.environ.get("DATA_DIR"):
    DS = Path(os.environ["DATA_DIR"])
elif ON_KAGGLE:
    DS = WORK / "yolo_v2"
    log(f"{assemble_dataset(DS)} dosya birleştirildi → {DS}")
else:
    DS = REPO / "furkan" / "outputs" / "yolo_v2"


def read_ids(name):
    return [Path(s).stem for s in (DS / name).read_text().split() if s.strip()]


val_ids, test_ids = read_ids("val.txt"), read_ids("test.txt")
assert len(val_ids) == 1295 and len(test_ids) == 2118, (len(val_ids), len(test_ids))
subsets = {n: read_ids(f"{n}.txt") for n in ["val_1400x788", "val_dark", "val_1400x788_dark"] if (DS / f"{n}.txt").exists()}
log(f"veri: {DS} | val {len(val_ids)} · test {len(test_ids)}")

In [ ]:
def image_info(folder, ids):
    rows = []
    for i in ids:
        with Image.open(DS / folder / f"{i}.jpg") as im:
            w, h_ = im.size
            im.draft("L", (128, 128))
            b = float(np.asarray(im.convert("L").resize((64, 64))).mean())
        rows.append((i, w, h_, f"{w}x{h_}" + (" · karanlık" if b < DARK_THR else " · aydınlık")))
    return pd.DataFrame(rows, columns=["image_id", "W", "H", "katman"]).set_index("image_id")


val_info, test_info = image_info("images/train", val_ids), image_info("images/test", test_ids)
p_test, p_val = test_info.katman.value_counts(normalize=True), val_info.katman.value_counts(normalize=True)
w_img = val_info.katman.map((p_test / p_val).fillna(0.0)).fillna(0.0)
w_img = w_img * len(w_img) / w_img.sum()
log(f"etkin örnek sayısı (ESS): {w_img.sum() ** 2 / (w_img ** 2).sum():.0f} / {len(w_img)}  (11m notebook'unda 1167)")

# Doğrulama kümesinin gerçek kutuları (piksel, sol üst köşe)
gt_rows = []
for i in val_ids:
    W, H = val_info.loc[i, ["W", "H"]]
    for s in (DS / "labels/train" / f"{i}.txt").read_text().splitlines():
        if s.strip():
            c, cx, cy, bw, bh = s.split()
            cx, cy, bw, bh = float(cx) * W, float(cy) * H, float(bw) * W, float(bh) * H
            gt_rows.append((i, CLASSES[int(c)], cx - bw / 2, cy - bh / 2, bw, bh))
gt = pd.DataFrame(gt_rows, columns=["image_id", "label", "x", "y", "w", "h"])
pd.crosstab(gt.label, "val kutu").reindex(CLASSES)

<a id="3"></a>
## 3 · Yarışma metriği ve tahmin fonksiyonları

- `evaluate`: Kaggle kuralının uygulaması. Sınıf bazında güvene göre sıralama, IoU ≥ 0,5 ile açgözlü eşleştirme, eşleşmemiş 200 px² altı tahminler yok sayılır, all-point AP, dört sınıfın ortalaması. İsteğe bağlı görüntü ağırlıkları alır.
- `predict`: görüntü görüntü tahmin (`conf=0.001`, NMS 0,70, `max_det=1000`, FP16). Görüntü kenarında sıfır boyuta kırpılmış kutular atılır.
- `nms`: kaydedilmiş kutulara sınıf bazında NMS uygular ve görüntü başına en yüksek güvenli `MAX_DET` kutuyu tutar. Hem eşik taramasında hem ensemble birleştirmesinde kullanılır.

In [ ]:
def _iou(b, B):
    x1 = np.maximum(b[0], B[:, 0]); y1 = np.maximum(b[1], B[:, 1])
    x2 = np.minimum(b[0] + b[2], B[:, 0] + B[:, 2]); y2 = np.minimum(b[1] + b[3], B[:, 1] + B[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    return inter / np.maximum(b[2] * b[3] + B[:, 2] * B[:, 3] - inter, 1e-9)


def evaluate(pred, ids, weights=None, min_area=200.0):
    """Yarışma tarzı mAP@0.5 (isteğe bağlı görüntü ağırlıklarıyla). yolo11m.ipynb ile birebir aynı."""
    ids = list(ids)
    w = pd.Series(1.0, index=ids) if weights is None else weights.reindex(ids).fillna(0.0)
    keep = set(w.index[w > 0])
    g_all, p_all = gt[gt.image_id.isin(keep)], pred[pred.image_id.isin(keep)]
    ap = {}
    for c in CLASSES:
        g = g_all[g_all.label == c]
        p = p_all[p_all.label == c].sort_values("conf", ascending=False, kind="mergesort")
        boxes = {k: v[["x", "y", "w", "h"]].to_numpy(float) for k, v in g.groupby("image_id")}
        used = {k: np.zeros(len(v), bool) for k, v in boxes.items()}
        npos = float(w.reindex(g.image_id).sum())
        if npos == 0:
            ap[c] = float("nan"); continue
        tp, fp = np.zeros(len(p)), np.zeros(len(p))
        pw = w.reindex(p.image_id).to_numpy()
        for k, (img, b) in enumerate(zip(p.image_id.to_numpy(), p[["x", "y", "w", "h"]].to_numpy(float))):
            B, j, best = boxes.get(img), -1, 0.0
            if B is not None:
                ious = _iou(b, B); ious[used[img]] = -1.0
                j = int(np.argmax(ious)); best = ious[j]
            if best >= 0.5:
                used[img][j] = True; tp[k] = pw[k]
            elif b[2] * b[3] >= min_area:
                fp[k] = pw[k]
        ctp, cfp = np.cumsum(tp), np.cumsum(fp)
        rec = np.concatenate([[0.0], ctp / npos, [1.0]])
        prec = np.concatenate([[0.0], ctp / np.maximum(ctp + cfp, 1e-12), [0.0]])
        prec = np.maximum.accumulate(prec[::-1])[::-1]
        idx = np.where(rec[1:] != rec[:-1])[0]
        ap[c] = float(np.sum((rec[idx + 1] - rec[idx]) * prec[idx + 1]))
    vals = [v for v in ap.values() if not np.isnan(v)]
    return {"mAP": float(np.mean(vals)) if vals else float("nan"), **ap}


PRED_KW = dict(conf=0.001, iou=0.7, max_det=MAX_DET, device=DEVICE,
               quantize=16 if torch.cuda.is_available() else None, verbose=False)   # FP16 on GPU (`half` is deprecated)
METHODS = {"A · 1280": ("a_1280", IMGSZ, False), "C · TTA@1536": ("c_tta1536", HI_IMGSZ, True)}
CLS_IDX = {c: k for k, c in enumerate(CLASSES)}


def predict(weights, folder, ids, imgsz, augment):
    """Görüntü görüntü tahmin (liste kaynağı Ultralytics'te tek batch sayıldığı için bellek taşar)."""
    m = YOLO(str(weights))
    rows = []
    for i in ids:
        r = m.predict(str(DS / folder / f"{i}.jpg"), imgsz=imgsz, augment=augment, **PRED_KW)[0]
        for (x1, y1, x2, y2), c, s in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
            if x2 - x1 < 1 or y2 - y1 < 1:      # görüntü kenarında sıfır boyuta kırpılmış kutu: geçersiz
                continue
            rows.append((i, CLASSES[int(c)], round(s, 5), round(x1, 1), round(y1, 1),
                         round(x2 - x1, 1), round(y2 - y1, 1)))
    return pd.DataFrame(rows, columns=["image_id", "label", "conf", "x", "y", "w", "h"])


def cached_predict(model, method, split):
    """Tahminleri preds/<model>_<yöntem>_<split>.csv olarak saklar; dosya varsa yeniden hesaplamaz."""
    slug, imgsz, augment = METHODS[method]
    path = PRED_DIR / f"{model}_{slug}_{split}.csv"
    if path.exists():
        return pd.read_csv(path, dtype={"image_id": str})
    t = time.time()
    folder, ids = ("images/train", val_ids) if split == "val" else ("images/test", test_ids)
    df = predict(WEIGHTS[model], folder, ids, imgsz, augment)
    df.to_csv(path, index=False)
    log(f"{model} · {method} · {split}: {len(df):,} kutu ({(time.time() - t) / 60:.1f} dk) → {path.name}")
    return df


def nms(df, thr, cap=MAX_DET):
    """Sınıf bazında NMS + görüntü başına en yüksek güvenli `cap` kutu. Ensemble'da birleştirme adımıdır."""
    if df.empty:
        return df
    keep_idx = []
    for _, g in df.groupby("image_id", sort=False):
        b = torch.tensor(g[["x", "y", "w", "h"]].to_numpy(), dtype=torch.float32)
        b[:, 2:] += b[:, :2]
        k = torchvision.ops.batched_nms(b, torch.tensor(g.conf.to_numpy(), dtype=torch.float32),
                                        torch.tensor(g.label.map(CLS_IDX).to_numpy()), thr).numpy()[:cap]
        keep_idx.extend(g.index[k])
    return df.loc[keep_idx]

<a id="4"></a>
## 4 · Modellerin kontrolü

Karşılaştırmanın geçerli olması için iki model de aynı sınıf sırasıyla ve **aynı train kümesiyle** (scene_holdout_v2 train, 5.176 görüntü) eğitilmiş olmalı.
Aksi halde val görüntüleri modellerden birinin eğitiminde yer almış olabilir. Checkpoint'lerin içindeki eğitim ayarları aşağıda listelenir.

In [ ]:
info = {}
for k, p in WEIGHTS.items():
    m = YOLO(str(p))
    assert m.names == {0: "car", 1: "van", 2: "truck", 3: "bus"}, f"{k}: sınıf sırası farklı: {m.names}"
    ta = (m.ckpt or {}).get("train_args", {})
    info[k] = {"dosya": Path(p).name, "MB": round(Path(p).stat().st_size / 2**20, 1),
               **{a: ta.get(a) for a in ["data", "imgsz", "epochs", "time", "optimizer", "cos_lr", "hsv_v", "flipud", "seed"]},
               "ultralytics": (m.ckpt or {}).get("version")}
pd.DataFrame(info)

**Kontrol:** `data` satırında beklenen dosyalar:

| Model | `data` | Dayanak |
|---|---|---|
| furkan | `data_v2.yaml` (train listesi `train_rfs.txt`) | — |
| hakan | `yolo/dataset.yaml` | Notebook gömülü `scene_holdout_v2` listesini yazıyor, 5.176 / 1.295 sayılarını assert ediyor |
| ardahan_l, ardahan_m_rfs | `/kaggle/working/data.yaml` | Script `scene_holdout_v2_train.txt` / `_val.txt` dosyalarını okuyor, 5.176 / 1.295 sayılarını ve ayrıklığı assert ediyor |

<a id="5"></a>
## 5 · Doğrulama tahminleri

Her model × yöntem için val tahmini üretilir. T4 veya RTX 3060'ta model başına A yaklaşık 1,5 dk, C yaklaşık 3 dk sürer.

In [ ]:
val_preds = {(m, meth): cached_predict(m, meth, "val") for m in WEIGHTS for meth in METHODS}
pd.DataFrame({f"{m} · {meth}": {"kutu": len(df), "görüntü başına": len(df) / len(val_ids)}
              for (m, meth), df in val_preds.items()}).T

<a id="6"></a>
## 6 · Tek model sonuçları

Her model × yöntem için NMS eşiği taranır. **Furkan · C · NMS 0,60 satırı 0,7894'ü tekrar üretmeli.** Bu satır evaluator ve veri hattının tutarlılık kontrolüdür.

In [ ]:
def score_row(name, df):
    sc = {"val (test ağırlıklı)": evaluate(df, val_ids, w_img), "val (ağırlıksız)": evaluate(df, val_ids)}
    for n, ids in subsets.items():
        sc[n] = evaluate(df, ids)
    return sc


candidates = {}          # aday adı → {"parts": [(model, yöntem), ...], "nms": eşik, "score": test ağırlıklı mAP}
sweep_rows = []
for (m, meth), df in val_preds.items():
    for thr in NMS_SWEEP:
        s = evaluate(nms(df, thr), val_ids, w_img)["mAP"]
        sweep_rows.append({"aday": f"{m} · {meth}", "NMS IoU": thr, "mAP": s})
    best = max((r for r in sweep_rows if r["aday"] == f"{m} · {meth}"), key=lambda r: r["mAP"])
    candidates[best["aday"]] = {"parts": [(m, meth)], "nms": best["NMS IoU"], "score": best["mAP"]}
    log(f"{best['aday']}: en iyi NMS {best['NMS IoU']} → test ağırlıklı mAP {best['mAP']:.4f}")

single_sweep = pd.DataFrame(sweep_rows).pivot(index="aday", columns="NMS IoU", values="mAP")
single_sweep

<a id="7"></a>
## 7 · Ensemble

İki modelin **aynı yöntemle** ürettiği kutular birleştirilir, üzerine sınıf bazında NMS uygulanır ve eşik taranır.
NMS, örtüşen iki kutudan güveni yüksek olanı tutar. Bu yüzden modellerin güven ölçekleri farklıysa (farklı optimizer ve kayıp dengesi) birleşim bir modele kayabilir.
Aşağıdaki güven dağılımı tablosu bu riski gösterir.

In [ ]:
conf_stats = pd.DataFrame({f"{m} · {meth}": df.conf.describe(percentiles=[.5, .9, .99])
                           for (m, meth), df in val_preds.items()}).T[["count", "mean", "50%", "90%", "99%"]]
display(conf_stats)

from itertools import combinations

ENS_METHOD = "C · TTA@1536"      # her modelin en iyi tek yöntemi; A ile ensemble denenmiyor
ens_rows = []
combos = [c for k in range(2, len(WEIGHTS) + 1) for c in combinations(WEIGHTS, k)] if ENSEMBLE else []
for combo in combos:
    merged = pd.concat([val_preds[(m, ENS_METHOD)] for m in combo], ignore_index=True)
    name = f"ensemble · {'+'.join(combo)} · {ENS_METHOD}"
    for thr in NMS_SWEEP:
        s = evaluate(nms(merged, thr), val_ids, w_img)["mAP"]
        ens_rows.append({"aday": name, "NMS IoU": thr, "mAP": s})
    best = max((r for r in ens_rows if r["aday"] == name), key=lambda r: r["mAP"])
    candidates[name] = {"parts": [(m, ENS_METHOD) for m in combo], "nms": best["NMS IoU"], "score": best["mAP"]}
    log(f"{name}: en iyi NMS {best['NMS IoU']} → test ağırlıklı mAP {best['mAP']:.4f}")

ens_sweep = (pd.DataFrame(ens_rows).pivot(index="aday", columns="NMS IoU", values="mAP") if ens_rows
             else pd.DataFrame())
ens_sweep if ENSEMBLE else log("tek model modu: ensemble atlandı")

<a id="8"></a>
## 8 · Karar

Tüm adaylar kendi en iyi NMS eşiğinde, sınıf ve alt küme kırılımıyla listelenir.
Ensemble'ın kazancı alt kümelerde de tutarlı mı, yoksa tek bir alt kümeden mi geliyor, bu tabloda görülür.

In [ ]:
def build(parts, thr, split):
    src = val_preds if split == "val" else {p: cached_predict(*p, "test") for p in parts}
    return nms(pd.concat([src[p] for p in parts], ignore_index=True), thr)


detail = {}
for name, c in candidates.items():
    sc = score_row(name, build(c["parts"], c["nms"], "val"))
    for k, v in sc.items():
        detail[(f"{name} · NMS {c['nms']}", k)] = v
detail = pd.DataFrame(detail).T
detail.index.names = ["aday", "küme"]

best_single = max((n for n in candidates if not n.startswith("ensemble")), key=lambda n: candidates[n]["score"])
best_ens = max((n for n in candidates if n.startswith("ensemble")), key=lambda n: candidates[n]["score"], default=None)
gain = candidates[best_ens]["score"] - candidates[best_single]["score"] if best_ens else float("nan")
final = best_ens if best_ens and gain >= MIN_GAIN else best_single

ranking = pd.DataFrame([{"aday": n, "NMS IoU": c["nms"], "test ağırlıklı mAP": c["score"]} for n, c in candidates.items()]
                       ).sort_values("test ağırlıklı mAP", ascending=False).set_index("aday")
display(ranking)
log(f"en iyi tek model: {best_single} ({candidates[best_single]['score']:.4f})")
if best_ens:
    log(f"en iyi ensemble:  {best_ens} ({candidates[best_ens]['score']:.4f}) · kazanç {gain:+.4f} (eşik {MIN_GAIN})")
log(f"SEÇİLEN: {final} · NMS {candidates[final]['nms']}")
detail

In [ ]:
# Katman bazında: en iyi ensemble ile en iyi tek modelin farkı (az görüntülü katmanlar gürültülüdür)
rows = []
for kat, ids in val_info.groupby("katman").groups.items():
    ids = list(ids)
    r = {"katman": kat, "görüntü": len(ids), "test payı %": p_test.get(kat, 0) * 100}
    for n in filter(None, [best_single, best_ens]):
        r[n] = evaluate(build(candidates[n]["parts"], candidates[n]["nms"], "val"), ids)["mAP"]
    if best_ens:
        r["fark"] = r[best_ens] - r[best_single]
    rows.append(r)
pd.DataFrame(rows).set_index("katman").sort_values("test payı %", ascending=False)

<a id="9"></a>
## 9 · Test tahmini ve submission

Hem **en iyi tek model** hem **en iyi ensemble** için submission yazılır. Hangisinin gönderileceğine Bölüm 8'deki karar kuralıyla karar verilir.
`submission.csv` seçilen adaydır. Format kontrolleri: her test görüntüsü için bir satır, tahmin yoksa `none`, `label conf x y w h` grupları, `0 ≤ conf ≤ 1`, `w, h > 0`.

In [ ]:
def write_submission(test_pred, path):
    s = (test_pred.label + " " + test_pred.conf.astype(str) + " " + test_pred.x.astype(str) + " "
         + test_pred.y.astype(str) + " " + test_pred.w.astype(str) + " " + test_pred.h.astype(str))
    strings = s.groupby(test_pred.image_id).agg(" ".join)
    ids = sorted(test_ids)
    sub = pd.DataFrame({"image_id": ids, "PredictionString": [strings.get(i, "none") for i in ids]})
    assert len(sub) == len(set(test_ids)) and sub.PredictionString.str.len().gt(0).all()
    for ps in sub.PredictionString:
        if ps != "none":
            t = ps.split()
            assert len(t) % 6 == 0
            for k in range(0, len(t), 6):
                assert t[k] in CLASSES and 0 <= float(t[k + 1]) <= 1 and float(t[k + 4]) > 0 and float(t[k + 5]) > 0
    sub.to_csv(path, index=False)
    return sub


def slug(name):
    return name.replace(" · ", "_").replace("@", "").replace(" ", "").lower().replace("ensemble", "ens")


written = {}
for n in dict.fromkeys(filter(None, [final, best_single, best_ens])):
    c = candidates[n]
    pt = build(c["parts"], c["nms"], "test")
    path = WORK / f"submission_{slug(n)}_nms{int(round(c['nms'] * 100))}.csv"
    sub = write_submission(pt, path)
    written[n] = path
    log(f"{path.name}: {len(sub)} satır, {int((sub.PredictionString == 'none').sum())} boş, {len(pt):,} kutu")

shutil.copy(written[final], WORK / "submission.csv")
json.dump({"final": final, "final_nms": candidates[final]["nms"], "min_gain": MIN_GAIN, "gain": gain,
           "candidates": {n: {"parts": [list(p) for p in c["parts"]], "nms": c["nms"], "score": c["score"]}
                          for n, c in candidates.items()},
           "single_sweep": {str(k): v for k, v in single_sweep.to_dict().items()},
           "ens_sweep": {str(k): v for k, v in ens_sweep.to_dict().items()},
           "weights": {k: str(v) for k, v in WEIGHTS.items()}},
          open(WORK / "compare_scores.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
log(f"submission.csv ← {final} · NMS {candidates[final]['nms']}")
pd.read_csv(WORK / "submission.csv").head()

**Çıktılar** (`/kaggle/working` veya `furkan/outputs/compare_11m/`, gitignored)

| Dosya | İçerik |
|---|---|
| `submission.csv` | Bölüm 8'in karar kuralıyla seçilen aday |
| `submission_<aday>_nms<eşik>.csv` | En iyi tek model ve en iyi ensemble submission'ları |
| `compare_scores.json` | Tüm adayların skorları, NMS taramaları, ağırlık yolları |
| `preds/*.csv` | Ham tahminler (`image_id,label,conf,x,y,w,h`, NMS 0,70) |

**Göndermeden önce:** `ardahan/check_submission.py` ile formatı doğrula. Bugüne kadarki val→LB farkı (11s: 0,731 → 0,64) hatırlanarak LB beklentisi val skorundan düşük tutulmalı.